# Stanford News: scrape article content

Reads `./Data/Stanford Articles Links.csv` and retrieves each story with Playwright + Chromium. Metadata comes from JSON-LD with HTML fallbacks.

Outputs: `./Data/Stanford Articles.csv`, `./Data/Stanford Article Errors.csv`




In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "playwright": "playwright",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "tqdm": "tqdm",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])


In [ ]:
import json
import random
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup, Tag
from playwright.async_api import async_playwright
from tqdm.auto import tqdm


In [ ]:
LINKS_PATH = Path("./Data/Stanford Articles Links.csv")
OUTPUT_PATH = Path("./Data/Stanford Articles.csv")
ERROR_PATH = Path("./Data/Stanford Article Errors.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

HEADLESS = False
NAVIGATION_TIMEOUT_MS = 60_000
MIN_DELAY = 0.8
MAX_DELAY = 1.5
CHECKPOINT_EVERY = 10

# Use an integer such as 3 for a test run; use None for all links.
MAX_ARTICLES = None


In [ ]:
links = pd.read_csv(LINKS_PATH)
links = links.loc[:, ~links.columns.str.startswith("Unnamed:")]

if "Link" not in links.columns:
    raise ValueError("The links CSV must contain a 'Link' column.")

links = (
    links.dropna(subset=["Link"])
    .drop_duplicates(subset=["Link"])
    .reset_index(drop=True)
)

if MAX_ARTICLES is not None:
    links = links.head(MAX_ARTICLES)

print(f"Articles to scrape: {len(links)}")
display(links.head())


In [ ]:
def clean_text(value):
    if value is None:
        return ""
    return " ".join(str(value).split())


def normalize_label(value):
    return clean_text(value).rstrip(":").casefold()


def normalize_date(value):
    text = clean_text(value)
    if not text:
        return ""

    text_without_ordinals = re.sub(r"(?<=\d)(st|nd|rd|th)\b", "", text, flags=re.I)
    parsed = pd.to_datetime(text_without_ordinals, errors="coerce", utc=True)
    if pd.isna(parsed):
        return text
    return parsed.strftime("%Y-%m-%d")


def iter_json_ld_objects(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from iter_json_ld_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_json_ld_objects(child)


def get_article_json_ld(soup):
    article_types = {
        "article",
        "newsarticle",
        "reportagenewsarticle",
        "analysisnewsarticle",
    }
    candidates = []

    for script in soup.find_all("script", type="application/ld+json"):
        raw = script.string or script.get_text()
        if not raw.strip():
            continue

        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue

        for obj in iter_json_ld_objects(data):
            obj_type = obj.get("@type", "")
            if isinstance(obj_type, list):
                normalized_types = {clean_text(item).casefold() for item in obj_type}
            else:
                normalized_types = {clean_text(obj_type).casefold()}

            if normalized_types & article_types:
                candidates.append(obj)

    if not candidates:
        return {}

    return max(candidates, key=lambda item: len(json.dumps(item, ensure_ascii=False)))


def meta_content(soup, *, name=None, property_name=None):
    if name:
        tag = soup.find("meta", attrs={"name": name})
    else:
        tag = soup.find("meta", attrs={"property": property_name})
    return clean_text(tag.get("content")) if tag and tag.get("content") else ""


def names_from_json_ld(value):
    if isinstance(value, str):
        return clean_text(value)
    if isinstance(value, dict):
        return clean_text(value.get("name"))
    if isinstance(value, list):
        names = [names_from_json_ld(item) for item in value]
        return "; ".join(name for name in names if name)
    return ""


def extract_labeled_value(soup, label):
    wanted = normalize_label(label)
    label_tags = []

    for tag in soup.find_all(["h2", "h3", "h4", "dt", "strong"]):
        if normalize_label(tag.get_text(" ", strip=True)) == wanted:
            label_tags.append(tag)

    for tag in reversed(label_tags):
        for sibling in tag.next_siblings:
            if isinstance(sibling, Tag) and sibling.name in {"h2", "h3", "h4", "dt"}:
                break
            text = (
                clean_text(sibling.get_text(" ", strip=True))
                if isinstance(sibling, Tag)
                else clean_text(sibling)
            )
            if text and normalize_label(text) != wanted:
                return text

        parent = tag.parent
        if isinstance(parent, Tag):
            next_tag = parent.find_next_sibling()
            if isinstance(next_tag, Tag):
                text = clean_text(next_tag.get_text(" ", strip=True))
                if text and normalize_label(text) != wanted:
                    return text

    return ""


In [ ]:
STOP_HEADINGS = {
    "for more information",
    "writer",
    "campus unit",
    "related topics",
    "share this story",
    "subscribe to stanford report",
    "stories for you",
    "popular stories",
    "read next",
}

BOILERPLATE_EXACT = {
    "copy link",
    "sign up",
    "previous",
    "next",
    "accept",
    "decline",
}


def is_stop_heading(text):
    normalized = normalize_label(text)
    return any(
        normalized == stop or normalized.startswith(f"{stop} ")
        for stop in STOP_HEADINGS
    )


def content_blocks_from_container(container, title="", subheading=""):
    title_norm = clean_text(title).casefold()
    subheading_norm = clean_text(subheading).casefold()

    matching_h1 = [
        tag
        for tag in container.find_all("h1")
        if not title_norm
        or clean_text(tag.get_text(" ", strip=True)).casefold() == title_norm
    ]
    start_marker = matching_h1[-1] if matching_h1 else None
    started = start_marker is None

    blocks = []
    seen = set()

    for node in container.descendants:
        if node is start_marker:
            started = True
            continue
        if not started or not isinstance(node, Tag):
            continue
        if node.name not in {"p", "li", "h2", "h3", "blockquote"}:
            continue

        if node.name in {"p", "li"}:
            selected_ancestor = node.find_parent(["p", "li", "blockquote"])
            if selected_ancestor is not None and selected_ancestor is not container:
                continue

        text = clean_text(node.get_text(" ", strip=True))
        if not text:
            continue
        if node.name in {"h2", "h3"} and is_stop_heading(text):
            break

        normalized = text.casefold()
        if normalized in BOILERPLATE_EXACT:
            continue
        if normalized == title_norm or normalized == subheading_norm:
            continue
        if re.search(r"\b\d+\s+min(?:ute)?s?\s+read\b", normalized):
            continue
        if normalized.startswith("media contact:"):
            continue
        if normalized.startswith("news, insights and events delivered"):
            continue
        if normalized in seen:
            continue

        seen.add(normalized)
        blocks.append(text)

    return blocks


def extract_article_body(soup, article_data, title, subheading):
    structured_body = article_data.get("articleBody")
    if isinstance(structured_body, str) and len(clean_text(structured_body)) >= 200:
        return clean_text(structured_body)

    selectors = [
        '[data-component*="story-body"]',
        '[class*="story-body"]',
        '[class*="article-body"]',
        '[class*="story-content"]',
        '[class*="article-content"]',
        ".entry-content",
        ".su-wysiwyg-text",
        ".wysiwyg",
        "main article",
        "article",
        "main",
    ]

    candidate_containers = []
    seen_ids = set()
    for selector in selectors:
        for container in soup.select(selector):
            container_id = id(container)
            if container_id not in seen_ids:
                seen_ids.add(container_id)
                candidate_containers.append(container)

    candidates = []
    for container in candidate_containers:
        blocks = content_blocks_from_container(container, title, subheading)
        body = "\n\n".join(blocks)
        if body:
            candidates.append(body)

    return max(candidates, key=len) if candidates else ""


In [ ]:
def parse_article_html(html, final_url):
    soup = BeautifulSoup(html, "html.parser")
    article_data = get_article_json_ld(soup)

    title = clean_text(article_data.get("headline"))
    if not title:
        main = soup.find("main") or soup
        headings = main.find_all("h1")
        title = clean_text(headings[-1].get_text(" ", strip=True)) if headings else ""

    subheading = clean_text(article_data.get("description"))
    if not subheading:
        subheading = (
            meta_content(soup, property_name="og:description")
            or meta_content(soup, name="description")
        )

    published_date = (
        article_data.get("datePublished")
        or meta_content(soup, property_name="article:published_time")
    )
    if not published_date:
        time_tag = soup.find("time")
        if time_tag:
            published_date = time_tag.get("datetime") or time_tag.get_text(" ", strip=True)

    author = extract_labeled_value(soup, "Writer")
    if not author:
        author = names_from_json_ld(article_data.get("author"))

    source = extract_labeled_value(soup, "Campus unit")
    if not source:
        source = names_from_json_ld(article_data.get("publisher")) or "Stanford Report"

    article_body = extract_article_body(soup, article_data, title, subheading)

    if not title:
        raise ValueError("Article title could not be extracted")
    if not article_body:
        raise ValueError("Article body could not be extracted")

    return {
        "Published Date": normalize_date(published_date),
        "Author": author,
        "Source": source,
        "Article Header": title,
        "Sub_Headings": subheading,
        "Article Body": article_body,
        "Url": final_url,
    }


In [ ]:
async def scrape_all_articles():
    rows = []
    errors = []

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(
            locale="en-US",
            viewport={"width": 1440, "height": 1200},
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/126.0.0.0 Safari/537.36"
            ),
        )

        # Text and metadata do not require images, video, or fonts.
        async def block_heavy_resources(route):
            if route.request.resource_type in {"image", "media", "font"}:
                await route.abort()
            else:
                await route.continue_()

        await context.route("**/*", block_heavy_resources)
        page = await context.new_page()
        page.set_default_navigation_timeout(NAVIGATION_TIMEOUT_MS)
        page.set_default_timeout(30_000)

        for index, url in enumerate(tqdm(links["Link"], desc="Articles"), start=1):
            try:
                response = await page.goto(url, wait_until="domcontentloaded")
                status = response.status if response is not None else None
                if status is not None and status >= 400:
                    raise RuntimeError(f"HTTP {status}")

                await page.wait_for_selector("main")
                row = parse_article_html(await page.content(), page.url)
                rows.append(row)

            except Exception as exc:
                errors.append({"Url": url, "Error": f"{type(exc).__name__}: {exc}"})
                print(f"Failed: {url}\n  {type(exc).__name__}: {exc}")

            if index % CHECKPOINT_EVERY == 0:
                pd.DataFrame(rows).to_csv(OUTPUT_PATH, index=False)
                pd.DataFrame(errors).to_csv(ERROR_PATH, index=False)

            await page.wait_for_timeout(int(random.uniform(MIN_DELAY, MAX_DELAY) * 1000))

        await context.close()
        await browser.close()

    return rows, errors


rows, errors = await scrape_all_articles()


In [ ]:
articles = pd.DataFrame(
    rows,
    columns=[
        "Published Date",
        "Author",
        "Source",
        "Article Header",
        "Sub_Headings",
        "Article Body",
        "Url",
    ],
)
article_errors = pd.DataFrame(errors, columns=["Url", "Error"])

articles.to_csv(OUTPUT_PATH, index=False)
article_errors.to_csv(ERROR_PATH, index=False)

print(f"Saved {len(articles)} articles to: {OUTPUT_PATH.resolve()}")
print(f"Recorded {len(article_errors)} errors in: {ERROR_PATH.resolve()}")
display(articles.head())


In [ ]:
print("Missing values by column:")
display(articles.isna().sum() + articles.eq("").sum())

if not article_errors.empty:
    display(article_errors.head(20))
